# 02_pipeline — config-driven orchestration template

Build, check, publish, and record evidence for governed Fabric data pipelines.

This notebook is intentionally thin and beginner friendly. For a normal one-source/one-target pipeline, edit the clearly marked **USER EDIT SECTION** blocks: source loading reads input DataFrames with the existing FabricOps IO helpers, source setup configures guardrails/evidence, the transform section creates target DataFrames, and target setup configures output tables and write behavior. To add another source or target table, add another DataFrame load and another dictionary to `SOURCE_TABLES` or `TARGET_TABLES`; do not copy profiling, schema, freshness, profile behavior, DQ, catalogue-evidence, write, lineage, or runtime-summary orchestration code.

FabricOps then enriches those source and target entries with guardrail defaults and write defaults before running profiling, schema validation, profile behavior enforcement, DQ enforcement, catalogue evidence, writes, lineage, and runtime summary from the config lists.

Flow:

1. Run `00_env_config`.
2. Import required functions.
3. Select the data agreement and capture run context.
4. Load source DataFrames with existing FabricOps read helpers.
5. Configure source guardrails and evidence in `SOURCE_TABLES`.
6. Review source guardrail defaults only when your project needs a different source governance policy.
7. Let FabricOps enrich source configs.
8. Run source guardrails before transformation.
9. Create target DataFrames in the transform section.
10. Configure output tables and write behavior in `TARGET_TABLES`.
11. Review target guardrail and write defaults only when your project needs a different target governance or write policy.
12. Let FabricOps add audit columns and enrich target configs.
13. Run target guardrails before writes.
14. Write targets only after all target guardrails pass.
15. Capture lineage and runtime summary evidence.

Schema, freshness, profile behavior, and DQ remain separate guardrail concepts. The reusable orchestration helper only removes repeated notebook code.


## 1. Run `00_env_config`

Load the shared FabricOps environment, path configuration, sample metadata, and metadata lakehouse routing.


In [ ]:
%run 00_env_config


## 2. Import required functions

The notebook imports existing FabricOps callables for agreement selection, table-config preparation, guardrail orchestration, explicit target writes, lineage, and runtime-summary evidence.


In [ ]:
from datetime import datetime, timezone

from fabricops_kit import (
    get_selected_agreement,
    prepare_pipeline_table_configs,
    read_lakehouse_csv,
    read_lakehouse_excel,
    read_lakehouse_parquet,
    read_lakehouse_table,
    read_warehouse_table,
    run_table_guardrails,
    widget_select_agreement,
    write_lakehouse_table,
    write_pipeline_lineage,
    write_pipeline_run_summary,
    write_warehouse_table,
)


## 3. Select data agreement and capture run context

Select the agreement that this pipeline satisfies. The selector registers this notebook in `METADATA_NOTEBOOK_REGISTRY` using the metadata target configured by `00_env_config`. The run context values are reused by guardrail evidence, lineage, and runtime summary writes.


In [ ]:
PIPELINE_STARTED_AT = datetime.now(timezone.utc).replace(microsecond=0).isoformat()
RUN_ID = RUN_CONTEXT.run_id
ENV_NAME = ENV
PIPELINE_NAME = "CHANGE_ME_pipeline"

widget_select_agreement(
    spark=spark,
    config=CONFIG,
    env=ENV_NAME,
    notebook_type="02_pipeline",
    pipeline_name=PIPELINE_NAME,
)

AGREEMENT = get_selected_agreement()
AGREEMENT_ID = AGREEMENT.get("agreement_id", "")
AGREEMENT_CONTRACT_VERSION = AGREEMENT.get("agreement_contract_version", AGREEMENT.get("contract_version", ""))
NOTEBOOK_REGISTRY_ID = AGREEMENT.get("notebook_registry_id", AGREEMENT.get("registration_id", ""))
NOTEBOOK_ID = AGREEMENT.get("notebook_id", RUN_CONTEXT.runtime_metadata.get("currentNotebookId", ""))


## 4. USER EDIT SECTION — load source DataFrames

Load each source DataFrame with the existing FabricOps read helper that matches where your data lives. The default starter path reads a Lakehouse table. To use a file or warehouse source, comment out the default read and uncomment the matching option below.


In [ ]:
df_source_01 = read_lakehouse_table(
    CONFIG,
    ENV_NAME,
    "source",
    "CHANGE_ME_source_table",
    spark_session=spark,
)

# CSV file in a configured Lakehouse target:
# df_source_01 = read_lakehouse_csv(
#     CONFIG,
#     ENV_NAME,
#     "source",
#     "CHANGE_ME/path/to/source.csv",
#     spark_session=spark,
#     header=True,
# )

# Parquet file or folder in a configured Lakehouse target:
# df_source_01 = read_lakehouse_parquet(
#     CONFIG,
#     ENV_NAME,
#     "source",
#     "CHANGE_ME/path/to/source.parquet",
#     verbose=True,
#     spark_session=spark,
# )

# Excel file in a configured Lakehouse target:
# df_source_01 = read_lakehouse_excel(
#     CONFIG,
#     ENV_NAME,
#     "source",
#     "CHANGE_ME/path/to/source.xlsx",
#     sheet_name=0,
#     spark_session=spark,
# )

# Warehouse table:
# df_source_01 = read_warehouse_table(
#     CONFIG,
#     ENV_NAME,
#     "source",
#     "dbo",
#     "CHANGE_ME_source_table",
#     spark_session=spark,
# )

# Custom Spark table reference:
# df_source_01 = spark.read.table("CHANGE_ME_database.CHANGE_ME_table")


## 5. USER EDIT SECTION — source guardrail configuration

Most users only edit this section for source guardrails and catalogue evidence. Update each source `table_name`, `watermark_column` when applicable, and `expected_schema`. FabricOps derives the governance `dataset_name` from `table_name`, uses `layer` as the default governance `stage`.

Valid FabricOps layer/stage concepts:

- `source` = raw/source lakehouse table.
- `unified` = cleaned/conformed lakehouse table.
- `product` = curated product or warehouse output.
- `metadata` = governance evidence lakehouse. It is normally configured in `00_env_config` and is not usually selected as a business source table.

For the default one-source pipeline, keep `key`, `df`, and `layer` as shown unless you know your pipeline uses a different configured layer. To support multiple source tables, load another source DataFrame above and add another dictionary to `SOURCE_TABLES` with a unique `key`. Do not copy profiling, schema, freshness, profile behavior, DQ, or catalogue-evidence code.

Advanced override support: add `dataset_name`, `stage`, or `dq_preset` inside a specific source table config only when that table needs to differ from the guardrail defaults.


In [ ]:
SOURCE_TABLES = [
    {
        "key": "source_01",
        "df": df_source_01,
        "layer": "source",
        "table_name": "CHANGE_ME_source_table",
        "watermark_column": "business_date",
        "expected_schema": {
            "customer_id": "bigint",
            "business_date": "date",
            "event_ts": "string",
            "status": "string",
            "amount": "double",
            "email": "string",
            "country_code": "string",
        },
    }
]

# To add a second source table, load df_source_02 above and add another dictionary to SOURCE_TABLES:
# {
#     "key": "source_02",
#     "df": df_source_02,
#     "layer": "source",
#     "table_name": "CHANGE_ME_second_source_table",
#     "watermark_column": "business_date",
#     "expected_schema": {"id": "bigint"},
# }
# Optional advanced per-table guardrail overrides, only when needed:
# "dataset_name": "CHANGE_ME_governance_dataset",
# "stage": "source",
# "dq_preset": "approved_rules",


## 6. Source guardrail defaults

These are the default guardrails applied to every source table. Most users should leave them as shown. Override a value inside a `SOURCE_TABLES` entry only when one source table needs a different schema rule, freshness rule, load behavior, DQ rule, profile distribution, or excluded column.


In [ ]:
DEFAULT_SOURCE_GUARDRAILS = {
    # Schema preset options:
    #   "allow_new_columns" = allow additive columns, block incompatible schema drift
    #   "strict" = require the schema to match exactly
    #   "monitor_only" = report schema differences without blocking
    "schema_preset": "allow_new_columns",

    # Load behavior guardrail options:
    #   "append" = protect existing history
    #   "overwrite" = accept full refresh/rebuild as the new state
    #   "skip" = skip only profile behavior enforcement
    "load_behavior": "append",

    # Freshness guardrail options:
    #   freshness_column = date/timestamp column that proves latest data arrived
    #   freshness_max_lag_days = allowed lag from today's date
    #   freshness_severity = "blocking" or "warning"
    "freshness_column": "business_date",
    "freshness_max_lag_days": 1,
    "freshness_severity": "blocking",

    # DQ preset options:
    #   "approved_rules" = enforce approved DQ rules from governance metadata
    #   "skip" = skip DQ enforcement for this table
    "dq_preset": "approved_rules",

    # Optional profile guardrail settings.
    "distribution_columns": [],
    "exclude_columns": None,
}


## 7. Prepare source table configs

FabricOps derives beginner-friendly governance defaults and adds the default source guardrails to each pre-loaded source DataFrame. Most users do not need to edit this section.


In [ ]:
SOURCE_TABLES, SOURCE_CONFIG_BY_KEY = prepare_pipeline_table_configs(
    SOURCE_TABLES,
    DEFAULT_SOURCE_GUARDRAILS,
    table_role="source",
)

# Convenience alias keeps the one-source starter transformation easy to read.
df_source_01 = SOURCE_CONFIG_BY_KEY["source_01"]["df"]


## 8. Optional: inspect a source schema

Run this cell while authoring if you want Spark to show the actual source schema before you finish `expected_schema` in the USER EDIT SECTION.


In [ ]:
# Optional authoring check: inspect the Spark schema before writing expected_schema.
df_source_01.printSchema()


## 9. Run source guardrails before transformation

FabricOps runs profiling, schema validation, profile behavior checks, DQ checks, catalogue evidence, and optional guardrail stopping through `run_table_guardrails`. Source guardrails run before transformation, and most users should not need to customize this orchestration code.


In [ ]:
source_guardrail_results = run_table_guardrails(
    SOURCE_TABLES,
    config=CONFIG,
    env=ENV_NAME,
    run_id=RUN_ID,
    spark_session=spark,
    agreement_id=AGREEMENT_ID,
    agreement_contract_version=AGREEMENT_CONTRACT_VERSION,
    notebook_registry_id=NOTEBOOK_REGISTRY_ID,
    notebook_id=NOTEBOOK_ID,
    pipeline_name=PIPELINE_NAME,
    stop_on_failure=True,
)

display(source_guardrail_results["summary"])

# Runtime summary and lineage cells reuse these package-generated evidence objects.
source_schema_results = source_guardrail_results["schema_results"]
source_freshness_results = source_guardrail_results["freshness_results"]
source_stability_results = source_guardrail_results["stability_results"]
source_dq_results = source_guardrail_results["dq_results"]
source_catalogue_status = source_guardrail_results["catalogue_status"]
source_evidence_definitions = source_guardrail_results["evidence_definitions"]


## 10. USER EDIT SECTION — DIY transformations

This is the only section where most users write business transformation logic. Create one target DataFrame for each target table you plan to publish. FabricOps guardrails and audit columns are handled in later sections.


In [ ]:
# DIY your transformations here.
# Replace this passthrough with your business logic.
df_target_01 = df_source_01

# Example:
# df_target_01 = (
#     df_source_01
#     .select("customer_id", "business_date", "event_ts", "status", "amount", "email", "country_code")
#     .where(F.col("amount").isNotNull())
#     .withColumn(
#         "amount_band",
#         F.when(F.col("amount") >= F.lit(100), F.lit("high"))
#         .when(F.col("amount") >= F.lit(25), F.lit("medium"))
#         .otherwise(F.lit("low")),
#     )
# )

# Add more transformations or joins here. For many sources, use SOURCE_CONFIG_BY_KEY.
# For many targets, create df_target_02, df_target_03, and reference each DataFrame in TARGET_TABLES below.


## 11. USER EDIT SECTION — target table configuration

Most users only edit this target section after creating target DataFrames in the transform section. Update each target `key`, `df`, `layer`, `table_name`, `write_mode`, `watermark_column`, and `expected_schema`. FabricOps derives the governance `dataset_name` from `table_name`, uses `layer` as the default governance `stage` and target write layer, uses `table_name` as the default target write name, uses `lakehouse` as the default target kind.

For the default one-target pipeline, keep `df` as the DataFrame created in the transform section. To support multiple target tables, add another dictionary to `TARGET_TABLES` with a unique `key` and a DataFrame created in the transform section. Do not copy profiling, schema, freshness, profile behavior, DQ, catalogue-evidence, or write orchestration code.

Advanced override support: add `dataset_name`, `stage`, `target_layer`, `target_name`, `target_kind`, `dq_preset`, or `kind` inside a specific target table config only when that table needs to differ from the guardrail or write defaults.


In [ ]:
TARGET_TABLES = [
    {
        "key": "target_01",
        "df": df_target_01,
        "layer": "unified",
        "table_name": "CHANGE_ME_target_table",
        "write_mode": "overwrite",
        "watermark_column": "business_date",
        "expected_schema": {
            "customer_id": "bigint",
            "business_date": "date",
            "event_ts": "string",
            "status": "string",
            "amount": "double",
            "email": "string",
            "country_code": "string",
            "_fabricops_run_id": "string",
            "_fabricops_pipeline_name": "string",
            "_fabricops_created_at": "string",
        },
    }
]

# To add a second target table, create df_target_02 in the transform section,
# then add another dictionary to TARGET_TABLES:
# {
#     "key": "target_02",
#     "df": df_target_02,
#     "layer": "product",
#     "table_name": "CHANGE_ME_second_target_table",
#     "write_mode": "overwrite",
#     "watermark_column": "business_date",
#     "expected_schema": {"id": "bigint"},
# }
# Optional advanced per-table overrides, only when needed:
# "dataset_name": "CHANGE_ME_governance_dataset",
# "stage": "product",
# "target_layer": "product",
# "target_name": "CHANGE_ME_written_table_name",
# "target_kind": "warehouse",
# "dq_preset": "approved_rules",
# "kind": "warehouse",
# "partition_by": ["business_date"],
# "repartition_by": ["customer_id"],
# "overwrite_schema": True,


## 12. Target guardrail and write defaults

These are the default guardrails and write options applied to every target table. Most users should leave them as shown. Override a value inside a `TARGET_TABLES` entry only when one target table needs a different schema rule, freshness rule, load behavior, DQ rule, profile distribution, excluded column, or write option.


In [ ]:
DEFAULT_TARGET_GUARDRAILS_AND_WRITE_OPTIONS = {
    # Schema preset options:
    #   "strict" = require the schema to match exactly
    #   "allow_new_columns" = allow additive columns, block incompatible schema drift
    #   "monitor_only" = report schema differences without blocking
    "schema_preset": "strict",

    # Load behavior guardrail options:
    #   "append" = protect existing history
    #   "overwrite" = accept full refresh/rebuild as the new state
    #   "skip" = skip only profile behavior enforcement
    "load_behavior": "overwrite",

    # Freshness guardrail options:
    #   freshness_column = date/timestamp column that proves latest data arrived
    #   freshness_max_lag_days = allowed lag from today's date
    #   freshness_severity = "blocking" or "warning"
    "freshness_column": "business_date",
    "freshness_max_lag_days": 1,
    "freshness_severity": "blocking",

    # DQ preset options:
    #   "approved_rules" = enforce approved DQ rules from governance metadata
    #   "skip" = skip DQ enforcement for this table
    "dq_preset": "approved_rules",

    # Optional profile guardrail settings.
    "distribution_columns": ["status", "amount", "country_code"],
    "exclude_columns": None,

    # Write mode options for Lakehouse targets:
    #   "overwrite" = replace the target table
    #   "append" = add rows to the target table
    #   "errorifexists" = fail if the target table already exists
    #   "ignore" = skip the write if the target table already exists
    # Warehouse writes use Spark connector modes such as "overwrite" or "append".
    "write_mode": "overwrite",

    # Optional Lakehouse write options.
    "partition_by": None,
    "repartition_by": None,
    "overwrite_schema": True,

    # Target kind options:
    #   "lakehouse" = write a Lakehouse Delta table
    #   "warehouse" = write a Fabric Warehouse table
    "kind": "lakehouse",
}


## 13. Prepare target table configs

Do not edit this section for normal target tables. FabricOps adds runtime audit columns, applies `DEFAULT_TARGET_GUARDRAILS_AND_WRITE_OPTIONS`, derives target write metadata, and keeps downstream guardrails and writes driven by `TARGET_TABLES`.

The `TARGET_01_*` variables created here are convenience aliases for starter notebook readability and lineage text. They are not user-edit fields.


In [ ]:
TARGET_TABLES, TARGET_CONFIG_BY_KEY = prepare_pipeline_table_configs(
    TARGET_TABLES,
    DEFAULT_TARGET_GUARDRAILS_AND_WRITE_OPTIONS,
    table_role="target",
    run_id=RUN_ID,
    pipeline_name=PIPELINE_NAME,
)

# Convenience aliases keep starter lineage text and one-target writes easy to read.
TARGET_01_CONFIG = TARGET_CONFIG_BY_KEY["target_01"]
TARGET_01_KEY = TARGET_01_CONFIG["key"]
TARGET_01_TABLE_NAME = TARGET_01_CONFIG["table_name"]
TARGET_01_LAYER = TARGET_01_CONFIG["layer"]
TARGET_01_WRITE_MODE = TARGET_01_CONFIG["write_mode"]


## 14. Run target guardrails before writes

Target profiling, schema validation, profile behavior checks, DQ checks, and catalogue evidence run for every config in `TARGET_TABLES`. Target writes do not happen unless `run_table_guardrails(..., stop_on_failure=True)` completes.


In [ ]:
target_guardrail_results = run_table_guardrails(
    TARGET_TABLES,
    config=CONFIG,
    env=ENV_NAME,
    run_id=RUN_ID,
    spark_session=spark,
    agreement_id=AGREEMENT_ID,
    agreement_contract_version=AGREEMENT_CONTRACT_VERSION,
    notebook_registry_id=NOTEBOOK_REGISTRY_ID,
    notebook_id=NOTEBOOK_ID,
    pipeline_name=PIPELINE_NAME,
    stop_on_failure=True,
)

display(target_guardrail_results["summary"])

target_schema_results = target_guardrail_results["schema_results"]
target_freshness_results = target_guardrail_results["freshness_results"]
target_stability_results = target_guardrail_results["stability_results"]
target_dq_results = target_guardrail_results["dq_results"]
target_catalogue_status = target_guardrail_results["catalogue_status"]
target_evidence_definitions = target_guardrail_results["evidence_definitions"]


## 15. Write target tables

Only after all configured target guardrails pass, explicitly write each target DataFrame to its configured Fabric target. Keep this section visible because it is the point where data is published.


In [ ]:
target_write_status = {}
for target_config in TARGET_TABLES:
    target_key = target_config["key"]
    target_kind = str(target_config.get("target_kind", target_config.get("kind", "lakehouse"))).lower()
    target_layer = target_config.get("target_layer", target_config.get("layer", "unified"))
    target_name = target_config.get("target_name", target_config.get("table_name", target_key))
    target_mode = target_config.get("write_mode", target_config.get("mode", "overwrite"))

    if target_kind == "lakehouse":
        write_lakehouse_table(
            target_config["df"],
            CONFIG,
            ENV_NAME,
            target_layer,
            target_name,
            mode=target_mode,
            partition_by=target_config.get("partition_by"),
            repartition_by=target_config.get("repartition_by"),
            overwrite_schema=target_config.get("overwrite_schema", target_mode == "overwrite"),
        )
    elif target_kind == "warehouse":
        write_warehouse_table(
            target_config["df"],
            CONFIG,
            ENV_NAME,
            target_layer,
            target_config.get("schema", "dbo"),
            target_name,
            mode=target_mode,
        )
    else:
        raise ValueError(f"Unsupported target kind for {target_key}: {target_kind}")

    target_write_status[target_key] = "written"


## 16. USER EDIT SECTION — lineage relationships

Describe how configured source tables produce configured target tables. Add another relationship dictionary when one pipeline writes multiple target tables or combines multiple sources.


In [ ]:
LINEAGE_RELATIONSHIPS = [
    {
        "sources": ["source_01"],
        "targets": [TARGET_01_KEY],
        "operation": f"publish {TARGET_01_TABLE_NAME}",
        "description": f"{SOURCE_CONFIG_BY_KEY['source_01']['table_name']} rows are published to {TARGET_01_TABLE_NAME}.",
    },
]


## 17. Write lineage

FabricOps writes lineage evidence after target writes complete.

In [ ]:
lineage_result = write_pipeline_lineage(
    spark=spark,
    config=CONFIG,
    env=ENV_NAME,
    run_id=RUN_ID,
    source_definitions=source_evidence_definitions,
    target_definitions=target_evidence_definitions,
    relationships=LINEAGE_RELATIONSHIPS,
    dataset_name=TARGET_01_CONFIG["dataset_name"],
    agreement_id=AGREEMENT_ID,
    agreement_contract_version=AGREEMENT_CONTRACT_VERSION,
    notebook_registry_id=NOTEBOOK_REGISTRY_ID,
    notebook_id=NOTEBOOK_ID,
    pipeline_name=PIPELINE_NAME,
)


## 18. Write runtime summary

Runtime evidence is stored in `METADATA_PIPELINE_RUNS` and displayed for operational support.


In [ ]:
catalogue_status = "written" if source_catalogue_status and target_catalogue_status else "not_written"

run_summary = write_pipeline_run_summary(
    spark=spark,
    config=CONFIG,
    env=ENV_NAME,
    run_id=RUN_ID,
    agreement_id=AGREEMENT_ID,
    agreement_contract_version=AGREEMENT_CONTRACT_VERSION,
    notebook_registry_id=NOTEBOOK_REGISTRY_ID,
    notebook_id=NOTEBOOK_ID,
    pipeline_name=PIPELINE_NAME,
    started_at=PIPELINE_STARTED_AT,
    completed_at=datetime.now(timezone.utc).replace(microsecond=0).isoformat(),
    status="completed",
    source_definitions=source_evidence_definitions,
    target_definitions=target_evidence_definitions,
    source_schema_results=source_schema_results,
    target_schema_results=target_schema_results,
    source_freshness_results=source_freshness_results,
    target_freshness_results=target_freshness_results,
    source_stability_results=source_stability_results,
    target_stability_results=target_stability_results,
    source_dq_results=source_dq_results,
    target_dq_results=target_dq_results,
    lineage_status=lineage_result.get("status", "unknown"),
    catalogue_status=catalogue_status,
    message="Pipeline completed and metadata evidence was written.",
)

display(run_summary)
